In [1]:
import torch
import torch.nn as nn

In [2]:
class Siec_5x5(nn.Module):
    def __init__(self, channels=16):
        super(Siec_5x5, self).__init__()
        # Jedna warstwa z dużym filtrem 5x5
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=5, bias=False)

    def forward(self, x):
        return self.conv1(x)

class Siec_Dwa_3x3(nn.Module):
    def __init__(self, channels=16):
        super(Siec_Dwa_3x3, self).__init__()
        # Dwie warstwy z małymi filtrami 3x3
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, bias=False)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, bias=False)

    def forward(self, x):
        x = self.conv1(x)
        print(f"   [Krok pośredni] Kształt po pierwszym splocie 3x3: {x.shape}")
        x = self.conv2(x)
        return x

In [3]:
print("--- DOWÓD 1: POLE WIDZENIA ---")
# Tworzymy "obrazek" o rozmiarze 5x5 pikseli (1 obrazek w batchu, 16 kanałów)
test_image = torch.randn(1, 16, 5, 5)
print(f"Oryginalny obrazek: {test_image.shape}\n")

--- DOWÓD 1: POLE WIDZENIA ---
Oryginalny obrazek: torch.Size([1, 16, 5, 5])



In [7]:
model_5x5 = Siec_5x5(channels=16)
model_3x3 = Siec_Dwa_3x3(channels=16)

In [9]:
print("Przepływ przez jeden filtr 5x5:")
out_5x5 = model_5x5(test_image)
print(f"-> Wynik końcowy: {out_5x5.shape}\n")

print("Przepływ przez dwa filtry 3x3:")
out_3x3 = model_3x3(test_image)
print(f"-> Wynik końcowy: {out_3x3.shape}\n")

Przepływ przez jeden filtr 5x5:
-> Wynik końcowy: torch.Size([1, 16, 1, 1])

Przepływ przez dwa filtry 3x3:
   [Krok pośredni] Kształt po pierwszym splocie 3x3: torch.Size([1, 16, 3, 3])
-> Wynik końcowy: torch.Size([1, 16, 1, 1])



In [13]:
if out_5x5.shape == out_3x3.shape:
    print("WNIOSEK 1: Obie sieci zredukowały obraz 5x5 do jednego piksela (1x1).")
    print("Oznacza to, że pojedynczy piksel na wyjściu w obu przypadkach 'widzi' dokładnie ten sam obszar 5x5!\n")

WNIOSEK 1: Obie sieci zredukowały obraz 5x5 do jednego piksela (1x1).
Oznacza to, że pojedynczy piksel na wyjściu w obu przypadkach 'widzi' dokładnie ten sam obszar 5x5!



In [15]:
print("--- DOWÓD 2: LICZBA PARAMETRÓW ---")

--- DOWÓD 2: LICZBA PARAMETRÓW ---


In [17]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

params_5x5 = count_parameters(model_5x5)
params_3x3 = count_parameters(model_3x3)

print(f"Liczba wag dla filtra 5x5: {params_5x5}")
print(f"Liczba wag dla dwóch filtrów 3x3: {params_3x3}")

zysk = params_5x5 - params_3x3
procent = (zysk / params_5x5) * 100

print(f"\nWNIOSEK 2: Dwa filtry 3x3 wykonują tę samą pracę przestrzenną,")
print(f"ale zużywają o {zysk} parametrów mniej (oszczędność rzędu {procent:.1f}%)!")
print("Dodatkowo zyskujemy możliwość wstawienia funkcji nieliniowej (np. ReLU) pomiędzy warstwy 3x3.")

Liczba wag dla filtra 5x5: 6400
Liczba wag dla dwóch filtrów 3x3: 4608

WNIOSEK 2: Dwa filtry 3x3 wykonują tę samą pracę przestrzenną,
ale zużywają o 1792 parametrów mniej (oszczędność rzędu 28.0%)!
Dodatkowo zyskujemy możliwość wstawienia funkcji nieliniowej (np. ReLU) pomiędzy warstwy 3x3.
